In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.7875, 10: 0.7825000000000001, 20: 0.7914999999999999, 30: 0.7954999999999999, 40: 0.8039999999999999, 50: 0.8105, 60: 0.8164999999999999, 70: 0.8175000000000001, 80: 0.8095000000000001, 90: 0.818, 100: 0.8150000000000001, 110: 0.8285, 120: 0.8314999999999999, 130: 0.8285, 140: 0.8295, 150: 0.8235000000000001, 160: 0.8314999999999999, 170: 0.8274999999999999, 180: 0.8294999999999998, 190: 0.8390000000000001, 200: 0.8340000000000002, 210: 0.8324999999999999, 220: 0.8309999999999998, 230: 0.8285, 240: 0.8320000000000001, 250: 0.8305, 260: 0.8348717948717947, 270: 0.835135135135135, 280: 0.8351351351351353, 290: 0.8362162162162164, 300: 0.8333333333333335}
{0: 0.01619375, 10: 0.012343749999999997, 20: 0.010817749999999998, 30: 0.00792975, 40: 0.007423999999999999, 50: 0.00839975, 60: 0.006717749999999999, 70: 0.0053837500000000005, 80: 0.00753975, 90: 0.007396, 100: 0.006254999999999998, 110: 0.00447775, 120: 0.00533775, 130: 0.005457749999999999, 140: 0.004939749999999999, 150: 0.00

In [4]:
mean_1_10, var_1_10 = summarize_by_step(root='.', start=1,  end=10)
mean_11_20, var_11_20 = summarize_by_step(root='.', start=11, end=20)
mean_21_30, var_21_30 = summarize_by_step(root='.', start=21, end=30)
mean_31_40, var_31_40 = summarize_by_step(root='.', start=31, end=40)

In [5]:
print(mean_1_10)
print(var_1_10)

{0: 0.868, 10: 0.85, 20: 0.8440000000000001, 30: 0.8320000000000001, 40: 0.8540000000000001, 50: 0.858, 60: 0.852, 70: 0.8440000000000001, 80: 0.85, 90: 0.8480000000000001, 100: 0.836, 110: 0.8480000000000001, 120: 0.866, 130: 0.8560000000000001, 140: 0.8380000000000001, 150: 0.8440000000000001, 160: 0.8640000000000001, 170: 0.85, 180: 0.8400000000000001, 190: 0.8480000000000001, 200: 0.8700000000000001, 210: 0.8480000000000001, 220: 0.8400000000000001, 230: 0.8459999999999999, 240: 0.8620000000000001, 250: 0.8480000000000001, 260: 0.8511111111111112, 270: 0.8533333333333333, 280: 0.8577777777777779, 290: 0.8511111111111112, 300: 0.8575}
{0: 0.0006559999999999996, 10: 0.0007400000000000002, 20: 0.0014240000000000001, 30: 0.0012959999999999992, 40: 0.0013640000000000008, 50: 0.001796000000000001, 60: 0.001136000000000001, 70: 0.0013439999999999995, 80: 0.0013000000000000002, 90: 0.0012160000000000007, 100: 0.0017440000000000003, 110: 0.0012959999999999998, 120: 0.0012040000000000006, 13

In [6]:
print(mean_11_20)
print(var_11_20)

{0: 0.844, 10: 0.8219999999999998, 20: 0.842, 30: 0.8560000000000001, 40: 0.8560000000000001, 50: 0.8740000000000002, 60: 0.8680000000000001, 70: 0.8619999999999999, 80: 0.8540000000000001, 90: 0.8640000000000001, 100: 0.86, 110: 0.8639999999999999, 120: 0.882, 130: 0.8840000000000001, 140: 0.876, 150: 0.8700000000000001, 160: 0.8720000000000001, 170: 0.866, 180: 0.8699999999999999, 190: 0.8779999999999999, 200: 0.884, 210: 0.8799999999999999, 220: 0.8720000000000001, 230: 0.8700000000000001, 240: 0.876, 250: 0.8659999999999999, 260: 0.8699999999999999, 270: 0.8800000000000001, 280: 0.8959999999999999, 290: 0.89, 300: 0.882}
{0: 0.0010240000000000008, 10: 0.0016359999999999997, 20: 0.0013160000000000001, 30: 0.0011040000000000004, 40: 0.0004640000000000008, 50: 0.0008040000000000014, 60: 0.0014560000000000026, 70: 0.0014760000000000012, 80: 0.0009640000000000006, 90: 0.0011840000000000004, 100: 0.001199999999999999, 110: 0.0015839999999999997, 120: 0.0014760000000000014, 130: 0.000704,

In [7]:
print(mean_21_30)
print(var_21_30)

{0: 0.8480000000000001, 10: 0.8460000000000001, 20: 0.8440000000000001, 30: 0.8320000000000001, 40: 0.834, 50: 0.8260000000000002, 60: 0.842, 70: 0.8440000000000001, 80: 0.8360000000000001, 90: 0.852, 100: 0.8400000000000001, 110: 0.8580000000000002, 120: 0.8380000000000001, 130: 0.8380000000000001, 140: 0.8459999999999999, 150: 0.8380000000000001, 160: 0.844, 170: 0.8480000000000001, 180: 0.842, 190: 0.8520000000000001, 200: 0.8240000000000001, 210: 0.8380000000000001, 220: 0.85, 230: 0.8380000000000001, 240: 0.844, 250: 0.8379999999999999, 260: 0.8460000000000001, 270: 0.8250000000000001, 280: 0.8200000000000001, 290: 0.8275000000000001, 300: 0.8250000000000001}
{0: 0.0029759999999999995, 10: 0.003443999999999999, 20: 0.002063999999999999, 30: 0.002256, 40: 0.0028839999999999985, 50: 0.0042439999999999995, 60: 0.003155999999999999, 70: 0.0015840000000000008, 80: 0.0029439999999999983, 90: 0.0027359999999999984, 100: 0.003199999999999998, 110: 0.002276, 120: 0.003075999999999998, 130:

In [8]:
print(mean_31_40)
print(var_31_40)

{0: 0.5900000000000001, 10: 0.6120000000000001, 20: 0.6359999999999999, 30: 0.662, 40: 0.6719999999999999, 50: 0.684, 60: 0.704, 70: 0.72, 80: 0.698, 90: 0.708, 100: 0.724, 110: 0.744, 120: 0.74, 130: 0.736, 140: 0.7580000000000001, 150: 0.742, 160: 0.746, 170: 0.746, 180: 0.766, 190: 0.778, 200: 0.758, 210: 0.764, 220: 0.762, 230: 0.76, 240: 0.7460000000000001, 250: 0.7699999999999999, 260: 0.774, 270: 0.782, 280: 0.7660000000000001, 290: 0.776, 300: 0.772}
{0: 0.007779999999999999, 10: 0.004336000000000002, 20: 0.006223999999999999, 30: 0.002916, 40: 0.0014559999999999996, 50: 0.004223999999999999, 60: 0.0039039999999999986, 70: 0.00424, 80: 0.008196000000000002, 90: 0.008176, 100: 0.0075039999999999985, 110: 0.003104, 120: 0.0034399999999999986, 130: 0.003744, 140: 0.0061959999999999975, 150: 0.006276000000000001, 160: 0.005924, 170: 0.006163999999999996, 180: 0.0032040000000000007, 190: 0.002436000000000001, 200: 0.004356000000000001, 210: 0.0036639999999999993, 220: 0.007236, 230: